### Nov 3, 2025
On staging:
- Create `/staging/pravindran/useful` if doesnt exist.

On local machine:
- `conda pack -n hytraitsenv -o hytraitsenv.tar.gz`
- `scp hytraitsenv.tar.gz pravindran@transfer.chtc.wisc.edu:/staging/pravindran/useful`

On staging:
- `ssh pravindran@transfer.chtc.wisc.edu`
- `cd /staging/pravindran`
- `./prep4project sophia-lakeview`: creates `/staging/pravindran/sophia-lakeview/io/<model, ideploy, edeploy>`


On local machine:
- While in `projworks` directory, run `./package4chtc.sh sophia-lakeview` to generate `CHTC/sophia-lakeview.zip`.
- While in `projworks` directory, transfer to staging on CHTC: `scp CHTC/sophia-lakeview.zip pravindran@transfer.chtc.wisc.edu:/staging/pravindran/sophia-lakeview/tonode`.


On Townsend submit node:
- `cd sophia-lakeview`.
- `python cleanup.py` - this creates empty `err`, `log` and `out` directories.
- Change contents of `submit_trait_train_internal_deploy.sub` for number of jobs.
- Change contents of `run_trait_train_internal_deploy.sh` for config parameters.
- `condor_submit submit_trait_train_internal_deploy.sub`

  
---

---

#### `package4chtc.sh`:

```
#!/bin/bash 

# cd into projworks before running this!

PROJECTNAME=$1

zip -rq hytraits.zip hytraits;
zip -rq $PROJECTNAME.zip $PROJECTNAME;

if [ -d CHTC/$PROJECTNAME ]; then
  rm -rf CHTC/$PROJECTNAME  
fi
  
mkdir CHTC/$PROJECTNAME;
mv hytraits.zip CHTC/$PROJECTNAME;
mv $PROJECTNAME.zip CHTC/$PROJECTNAME;
cp CHTC/common/hytraitsenv.tar.gz CHTC/$PROJECTNAME;

cd CHTC;
zip -rq $PROJECTNAME.zip $PROJECTNAME;
rm -rf $PROJECTNAME;
cd ..;
```

---

#### `submit_trait_train_internal_deploy.sub`:

```
# submit_trait_train_internal_deploy.sub
universe = vanilla

# IMPORTANT! Require execute servers that have Staging:
Requirements = (Target.HasCHTCStaging == true)

# Set files to capture log, standard output & error
log = $(Cluster)_$(Process).log
error = $(Cluster)_$(Process).err
output = $(Cluster)_$(Process).out

# Specify executable
executable = ./run_trait_train_internal_deploy.sub.sh

# Arguments to pass to the executable
arguments = $(Process)

should_transfer_files = YES
getenv = TRUE

request_cpus = 1
request_disk = 16GB
request_memory = 8GB

queue 1 # change to required number of jobs
```

---

#### `run_trait_train_internal_deploy.sh

```
#!/bin/bash

USERNAME=pravindran
PROJECTNAME=sophia-lakeview
ENVNAME=hytraitsenv
CONFIGIDX="$1"

# transfer project from staging
echo 'Transferring: staging --> remote ...'
cp /staging/$USERNAME/$PROJECTNAME/tonode/$PROJECTNAME.zip $HOME

# extract project in $HOME
echo "Extracting project package ..."
unzip -qq $HOME/$PROJECTNAME.zip -d $HOME
rm -f $HOME/$PROJECTNAME.zip 
mv $HOME/$PROJECTNAME/$ENVNAME.tar.gz $HOME
mv $HOME/$PROJECTNAME/$PROJECTNAME.zip $HOME 
mv $HOME/$PROJECTNAME/hytraits.zip $HOME
rm -rf $HOME/$PROJECTNAME

# hytraits
echo "--- Extracting hytraits.zip ..."
unzip -qq $HOME/hytraits.zip -d $HOME
rm -f $HOME/hytraits.zip

echo "--- Extracting ${PROJECTNAME} ..."
unzip -qq $HOME/$PROJECTNAME.zip -d $HOME
rm -f $HOME/$PROJECTNAME.zip

# conda environment
echo "--- Extracing hytraitsenv ..."
mkdir $HOME/$ENVNAME
tar -xzf $HOME/$ENVNAME.tar.gz -C $HOME/$ENVNAME
rm $HOME/$ENVNAME.tar.gz
export PATH=$HOME/$ENVNAME:$HOME/$ENVNAME/lib:$HOME/$ENVNAME/share:$PATH
source $HOME/$ENVNAME/bin/activate
conda-unpack

echo "---------------------------------------------"

# process
echo "Processing ..."
# change next line appropriately.
python $HOME/$PROJECTNAME/setup.py --n_inners 50 --n_outers 200 --internal_deploy
python $HOME/$PROJECTNAME/train.py --config_idx "${CONFIGIDX}"
python $HOME/$PROJECTNAME/ideploy.py --config_idx "${CONFIGIDX}"

echo "---------------------------------------------"

# zip files for transfer to staging
MODELSDIR=$HOME/$PROJECTNAME/io/model
if [ -d "$MODELSDIR" ]; then
    for modeldir in "$MODELSDIR"/*/ ; do
        if [ -d "$modeldir" ]; then
            name=$(basename $modeldir)
            echo "Zipping model: ${name} ..."
            zip -qr $MODELSDIR/$name.zip $modeldir
	    echo "Transferring remote -> staging: ${name}.zip (model) ..."
            cp $MODELSDIR/$name.zip /staging/$USERNAME/$PROJECTNAME/fromnode/io/model
        fi
    done
fi


IDEPLOYSDIR=$HOME/$PROJECTNAME/io/deploy/ideploy
if [ -d "$IDEPLOYSDIR" ]; then
    for ideploydir in "$IDEPLOYSDIR"/*/ ; do
        if [ -d "$ideploydir" ]; then
            name=$(basename $ideploydir)
            echo "Zipping ideploy: ${name} ..."
            zip -qr $IDEPLOYSDIR/$name.zip $ideploydir
	    echo "Transferring remote -> staging: ${name}.zip (ideploy) ..."
	    cp $IDEPLOYSDIR/$name.zip /staging/$USERNAME/$PROJECTNAME/fromnode/io/deploy/ideploy
        fi
    done
fi

exit
exit
```